# Training

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load your data
df = pd.read_pickle("../data/processed/preprocessed_weather.pkl")

# --- Preprocessing ---

# Convert time to useful numeric features
df["time"] = pd.to_datetime(df["time"])
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day

# Drop original time column
df = df.drop(columns=["time"])

# Convert categorical to numeric (one-hot encoding)
df = pd.get_dummies(df, columns=["weather_code"], drop_first=True)

df.head()

,temperature_2m_max,temperature_2m_min,temperature_2m_mean,apparent_temperature_max,apparent_temperature_min,relative_humidity_2m_max,relative_humidity_2m_min,relative_humidity_2m_mean,wind_speed_10m_max,wind_gusts_10m_max,...,day,weather_code_1,weather_code_2,weather_code_3,weather_code_51,weather_code_53,weather_code_55,weather_code_61,weather_code_63,weather_code_65
0,21.8,6.5,14.3,20.9,4.3,94,39,68,4.9,11.4,...,1,False,False,False,False,False,False,False,False,False
1,22.6,8.0,14.6,21.7,6.0,95,41,71,6.4,9.8,...,2,False,False,False,False,False,False,False,False,False
2,22.6,7.8,14.6,21.2,6.2,97,41,73,6.5,13.4,...,3,False,False,False,False,False,False,False,False,False
3,22.2,8.0,14.7,21.6,6.1,96,35,69,5.8,10.3,...,4,False,False,True,False,False,False,False,False,False
4,23.4,9.9,16.0,21.5,8.2,86,42,66,7.9,16.6,...,5,False,False,True,False,False,False,False,False,False


## Linear Regression

In [5]:
# Set up training loop with Weights and Biases

from sklearn.linear_model import LinearRegression, Ridge, Lasso
import wandb

training_params = [
    # --- LinearRegression: only knob worth turning is fit_intercept ---
    {
        "run_name": "linear_reg_no_intercept",
        "model_class": LinearRegression,
        "params": {"fit_intercept": False}
    },
    {
        "run_name": "linear_reg_with_intercept",
        "model_class": LinearRegression,
        "params": {"fit_intercept": True}
    },

    # --- Ridge: L2 regularization, sweep alpha (regularization strength) ---
    {
        "run_name": "ridge_alpha_0.1",
        "model_class": Ridge,
        "params": {"alpha": 0.1, "fit_intercept": True}
    },
    {
        "run_name": "ridge_alpha_1.0",
        "model_class": Ridge,
        "params": {"alpha": 1.0, "fit_intercept": True}
    },
    {
        "run_name": "ridge_alpha_10.0",
        "model_class": Ridge,
        "params": {"alpha": 10.0, "fit_intercept": True}
    },

    # --- Lasso: L1 regularization, also sweeps alpha ---
    {
        "run_name": "lasso_alpha_0.1",
        "model_class": Lasso,
        "params": {"alpha": 0.1, "fit_intercept": True, "max_iter": 5000}
    },
    {
        "run_name": "lasso_alpha_1.0",
        "model_class": Lasso,
        "params": {"alpha": 1.0, "fit_intercept": True, "max_iter": 5000}
    },
]

# Prepare data
X = df.drop(columns=["apparent_temperature_max"], axis=1)
y = df["apparent_temperature_max"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Run each of the params with Weights and Biases
for config in training_params:
    run_name = config["run_name"]
    params = config["params"]
    model_class = config["model_class"]

    with wandb.init(project="weather-prediction", name=run_name, config=params) as run:
        model = model_class(**params)   # <-- use model_class here
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        run.log({"mse": mse, "r2": r2})

mse,▁
r2,▁
mse,0.45607
r2,0.99224


mse,▁
r2,▁
mse,0.45953
r2,0.99219


mse,▁
r2,▁
mse,0.45962
r2,0.99218


mse,▁
r2,▁
mse,0.4607
r2,0.99217


mse,▁
r2,▁
mse,0.47092
r2,0.99199


mse,▁
r2,▁
mse,0.49508
r2,0.99158


mse,▁
r2,▁
mse,0.94337
r2,0.98396


## KNN Regressor

In [6]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import wandb

knn_training_params = [
    # --- Sweep n_neighbors with uniform weighting ---
    {"run_name": "knn_k3_uniform", "params": {"n_neighbors": 3, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k5_uniform", "params": {"n_neighbors": 5, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k10_uniform", "params": {"n_neighbors": 10, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k20_uniform", "params": {"n_neighbors": 20, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k50_uniform", "params": {"n_neighbors": 50, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},

    # --- Sweep n_neighbors with distance weighting ---
    {"run_name": "knn_k3_distance", "params": {"n_neighbors": 3, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k5_distance", "params": {"n_neighbors": 5, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k10_distance", "params": {"n_neighbors": 10, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k20_distance", "params": {"n_neighbors": 20, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k50_distance", "params": {"n_neighbors": 50, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},

    # --- Sweep distance metrics ---
    {"run_name": "knn_k10_manhattan", "params": {"n_neighbors": 10, "weights": "distance", "metric": "manhattan", "algorithm": "auto"}},
    {"run_name": "knn_k10_chebyshev", "params": {"n_neighbors": 10, "weights": "distance", "metric": "chebyshev", "algorithm": "auto"}},
    {"run_name": "knn_k10_minkowski_p3", "params": {"n_neighbors": 10, "weights": "distance", "metric": "minkowski", "metric_params": {"p": 3}, "algorithm": "auto"}},

    # --- Sweep algorithms ---
    {"run_name": "knn_k10_balltree", "params": {"n_neighbors": 10, "weights": "distance", "metric": "euclidean", "algorithm": "ball_tree"}},
    {"run_name": "knn_k10_kdtree", "params": {"n_neighbors": 10, "weights": "distance", "metric": "euclidean", "algorithm": "kd_tree"}},
]

# KNN-specific training loop (with scaling pipeline)
for config in knn_training_params:
    run_name = config["run_name"]
    params = config["params"]

    with wandb.init(project="weather-prediction", name=run_name, config=params) as run:

        # Build pipeline — scaler is essential for KNN
        knn_model = KNeighborsRegressor(**params)
        
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", knn_model)
        ])

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        run.log({"mse": mse, "r2": r2})

mse,▁
r2,▁
mse,4.34275
r2,0.92615


mse,▁
r2,▁
mse,4.19748
r2,0.92862


mse,▁
r2,▁
mse,4.32482
r2,0.92645


mse,▁
r2,▁
mse,5.07167
r2,0.91375


mse,▁
r2,▁
mse,6.04382
r2,0.89722


...

mse,▁
r2,▁
mse,10.81915
r2,0.81601


/workspaces/WeatherML/.venv/lib/python3.12/site-packages/sklearn/neighbors/_regression.py:227: SyntaxWarning: Parameter p is found in metric_params. The corresponding parameter from __init__ is ignored.
  return self._fit(X, y)


mse,▁
r2,▁
mse,5.22582
r2,0.91113


mse,▁
r2,▁
mse,3.95874
r2,0.93268


mse,▁
r2,▁
mse,3.95874
r2,0.93268
